In [1]:
%load_ext autoreload
%autoreload 2


In [5]:
# load the documents
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(r"/Users/saurabshrestha/Downloads/AMR Reports/untitled folder").load_data()


In [6]:
from ragas.testset import TestsetGenerator
from llama_index.llms.ollama import Ollama
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

generator_llm = Ollama(model="gemma3:4b")
# critic_llm =  GoogleGenAI(model="gemini-2.0-flash")
embeddings = OllamaEmbedding(model_name='mxbai-embed-large')

from ragas.testset import TestsetGenerator

generator = TestsetGenerator.from_llama_index(
    llm=generator_llm,
    embedding_model=embeddings,
)
dataset = generator.generate_with_llamaindex_docs(documents, testset_size=5)

Applying HeadlinesExtractor:   2%|▏         | 1/59 [03:52<3:45:09, 232.93s/it]unable to apply transformation: llama runner process no longer running: 2 
unable to apply transformation: health resp: Get "http://127.0.0.1:65482/health": dial tcp 127.0.0.1:65482: connect: connection refused
unable to apply transformation: llama runner process no longer running: 2 
Applying HeadlinesExtractor:   7%|▋         | 4/59 [04:12<45:21, 49.48s/it]   unable to apply transformation: llama runner process no longer running: 2 
unable to apply transformation: llama runner process no longer running: 2 
Applying HeadlinesExtractor:  10%|█         | 6/59 [04:32<28:28, 32.23s/it]unable to apply transformation: llama runner process no longer running: 2 
unable to apply transformation: llama runner process no longer running: 2 
unable to apply transformation: 
Applying HeadlinesExtractor:  17%|█▋        | 10/59 [05:05<14:32, 17.81s/it]unable to apply transformation: llama runner process no longer running: 2 

ValueError: No nodes that satisfied the given filer. Try changing the filter.

In [1]:
import os
from typing import Dict, List, Optional, Set
import fitz  # PyMuPDF
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter
from llama_index.core.readers.base import BaseReader
from llama_index.core.schema import Document
import unicodedata
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from dataclasses import dataclass
from collections import defaultdict

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.base_models import InputFormat

@dataclass
class TextBlock:
    text: str
    page_num: int
    block_index: int
    metadata: Dict = None


class TextMatcher:
    def __init__(self, similarity_threshold: float = 0.85, overlap_threshold: float = 0.6):
        self.similarity_threshold = similarity_threshold
        self.overlap_threshold = overlap_threshold
        self.vectorizer = TfidfVectorizer(
            analyzer='word',
            ngram_range=(1, 2),
            min_df=1,
            strip_accents='unicode'
        )
    
    def preprocess_text(self, text: str) -> str:
        """Enhanced text preprocessing."""
        text = unicodedata.normalize('NFKC', text)
        text = re.sub(r'-+\n', ' ', text) 
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s]', '', text)
        return text.strip().lower()
    
    def calculate_text_overlap(self, text1: str, text2: str) -> float:
        """Calculate character-level overlap between two texts."""
        text1_chars = set(self.preprocess_text(text1))
        text2_chars = set(self.preprocess_text(text2))
        overlap = len(text1_chars.intersection(text2_chars))
        total = len(text1_chars.union(text2_chars))
        return overlap / total if total > 0 else 0

    def fill_small_gaps(self,  pages: List[int], max_gap: int = 2) -> List[int]:
        if not pages:
            return []
        filled = set()
        for i in range(len(pages) - 1):
            filled.update(range(pages[i], pages[i+1] + 1) if pages[i+1] - pages[i] <= max_gap else [pages[i]])
        filled.add(pages[-1])
        return sorted(filled)

    def find_page_ranges(self, chunk_text: str, text_blocks: List[TextBlock]) -> List[int]:
        """Find page ranges for a chunk using multiple matching strategies."""
        chunk_text = self.preprocess_text(chunk_text)
        
        # Strategy 1: Exact substring matching (highest priority)
        matched_pages = set()
        chunk_sentences = chunk_text.split('.')
        for sentence in chunk_sentences:
            if len(sentence.strip()) < 20:  # Skip very short sentences
                continue
            for block in text_blocks:
                if sentence.strip() in self.preprocess_text(block.text):
                    matched_pages.add(block.page_num)
        
        if matched_pages:
            # Fill gaps in page ranges
            page_list = sorted(matched_pages)
            if len(page_list) > 1:
                matched_pages = set(self.fill_small_gaps(page_list))
            return sorted(matched_pages)
        
        # Strategy 2: TF-IDF similarity + overlap threshold with sliding window
        block_texts = [self.preprocess_text(block.text) for block in text_blocks]
        if not block_texts or not chunk_text:
            return [1]

        # Create overlapping windows of text to better match chunks that cross page boundaries
        window_size = 2
        windowed_texts = []
        windowed_pages = []
        for i in range(len(block_texts)):
            window_text = ' '.join(block_texts[max(0, i - window_size):min(len(block_texts), i + window_size + 1)])
            windowed_texts.append(window_text)
            windowed_pages.append(text_blocks[i].page_num)

        tfidf_matrix = self.vectorizer.fit_transform(windowed_texts + [chunk_text])
        similarities = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1])[0]

        # Find matches that pass BOTH similarity and overlap thresholds
        matched_indices = []
        for idx, sim in enumerate(similarities):
            if sim >= self.similarity_threshold:
                overlap = self.calculate_text_overlap(chunk_text, windowed_texts[idx])
                if overlap >= self.overlap_threshold:
                    matched_indices.append(idx)

        if matched_indices:
            matched_pages = {windowed_pages[i] for i in matched_indices}
            page_list = sorted(matched_pages)
            if len(page_list) > 1:
                full_range = set(range(min(page_list), max(page_list) + 1))
                matched_pages.update(full_range)
            return sorted(matched_pages)

        # Fallback: best match and neighbors
        best_match_idx = int(np.argmax(similarities))
        matched_pages = {windowed_pages[best_match_idx]}
        if best_match_idx > 0:
            matched_pages.add(windowed_pages[best_match_idx - 1])
        if best_match_idx < len(windowed_pages) - 1:
            matched_pages.add(windowed_pages[best_match_idx + 1])

        return sorted(matched_pages)


class EnhancedPDFReader(BaseReader):
    """Enhanced PDF Reader that leverages both pymupdf4llm for header detection
    and docling for high-quality markdown conversion."""

    HEADERS_TO_SPLIT_ON = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    def __init__(self, similarity_threshold: float = 0.9, chunk_size: int = 512):
        self.similarity_threshold = similarity_threshold
        self.text_matcher = TextMatcher(similarity_threshold=similarity_threshold)
        
        # Configure docling pipeline options
        pipeline_options = PdfPipelineOptions(
            do_table_structure=True,
            do_formula_enrichment=True,
            do_picture_description=True,
            table_structure_options=dict(
                mode=TableFormerMode.FAST
            ),
            enable_remote_services=False
        )
        
        # Set up docling document converter
        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
            }
        )
        
        # Set up markdown splitter
        self.markdown_splitter = MarkdownHeaderTextSplitter(
            self.HEADERS_TO_SPLIT_ON, 
            strip_headers=False
        )

    def _extract_pdf_metadata(self, doc: fitz.Document) -> Dict:
        """Extract comprehensive metadata from PDF."""
        metadata = {
            'title': doc.metadata.get('title', ''),
            'author': doc.metadata.get('author', ''),
            'subject': doc.metadata.get('subject', ''),
            'keywords': doc.metadata.get('keywords', ''),
            'total_pages': len(doc),
        }
        
        # Add creation and modification dates if available
        if doc.metadata.get('creationDate'):
            metadata['creation_date'] = doc.metadata['creationDate']
        if doc.metadata.get('modDate'):
            metadata['modification_date'] = doc.metadata['modDate']
            
        return metadata

    def _extract_page_metadata(self, page: fitz.Page) -> Dict:
        """Extract metadata for a specific page."""
        return {
            'page_number': page.number + 1,
            'width': page.rect.width,
            'height': page.rect.height,
            'rotation': page.rotation,
            'has_images': len(page.get_images()) > 0,
            'has_tables': bool(page.find_tables()), # if using PyMuPDF >= 1.23.0
        }

    def _extract_block_metadata(self, block: tuple, page_num: int) -> Dict:
        """Extract metadata from a text block."""
        x0, y0, x1, y1, text, block_type, block_no = block
        return {
            'block_type': block_type,
            'block_number': block_no,
            'position': {
                'x0': x0, 'y0': y0,
                'x1': x1, 'y1': y1
            },
            'location': 'top' if y0 < 100 else 'bottom' if y1 > 700 else 'middle',
            'is_header': y0 < 100 and len(text.strip()) < 200,
            'is_footer': y1 > 700 and len(text.strip()) < 200,
        }

    def _extract_pdf_text_with_pages(self, pdf_path: str) -> List[TextBlock]:
        """Extracts text and metadata from a PDF while preserving page numbers."""
        text_blocks = []
        doc = fitz.open(pdf_path)
        pdf_metadata = self._extract_pdf_metadata(doc)
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            page_metadata = self._extract_page_metadata(page)
            blocks = page.get_text("blocks")
            
            for idx, block in enumerate(blocks):
                block_metadata = self._extract_block_metadata(block, page_num)
                text_blocks.append(
                    TextBlock(
                        text=block[4].strip(),
                        page_num=page_num + 1,
                        block_index=idx,
                        metadata={
                            **pdf_metadata,
                            **page_metadata,
                            **block_metadata
                        }
                    )
                )
        doc.close()
        return text_blocks

    def _extract_header_info_from_pymupdf4llm(self, pdf_path: str) -> Dict:
        """Extract headers information using pymupdf4llm."""
        try:
            # Get the markdown with header information from pymupdf4llm
            pymupdf_md = pymupdf4llm.to_markdown(pdf_path)
            
            # Extract headers and their content
            header_info = {}
            current_header = None
            header_content = []
            
            for line in pymupdf_md.split('\n'):
                if line.startswith('#'):
                    # If we have a previous header, save it
                    if current_header:
                        header_info[current_header] = '\n'.join(header_content)
                        header_content = []
                    
                    # Start new header
                    current_header = line
                else:
                    if current_header:
                        header_content.append(line)
            
            # Save the last header
            if current_header:
                header_info[current_header] = '\n'.join(header_content)
                
            return header_info
        except Exception as e:
            print(f"Error extracting headers with pymupdf4llm: {e}")
            return {}

    def _convert_with_docling(self, pdf_path: str) -> str:
        """Convert PDF to markdown using docling."""
        try:
            result = self.converter.convert(pdf_path)
            return result.document.export_to_markdown()
        except Exception as e:
            print(f"Error converting with docling: {e}")
            return ""

    def _merge_markdown_with_headers(self, docling_md: str, header_info: Dict) -> str:
        """Merge docling markdown with pymupdf4llm headers."""
        if not header_info:
            return docling_md
            
        # Simple approach: replace header sections in docling_md with ones from pymupdf4llm
        result_md = docling_md
        
        for header, content in header_info.items():
            # Extract the header level and text
            header_parts = header.split(' ', 1)
            if len(header_parts) < 2:
                continue
                
            header_level = header_parts[0]  # e.g., "#", "##"
            header_text = header_parts[1]   # The header text
            
            # Create regex pattern to find this header section in docling markdown
            pattern = f"{re.escape(header_level)} {re.escape(header_text)}.*?(?=^#|\\Z)"
            
            # Replace the section
            replacement = f"{header}\n{content}\n\n"
            result_md = re.sub(pattern, replacement, result_md, flags=re.DOTALL | re.MULTILINE)
            
        return result_md

    def load_data(self, pdf_path: str, extra_info: Optional[Dict] = None) -> List[Document]:
        """Load data from PDF using both pymupdf4llm and docling for optimal results."""
        # Extract text blocks with page information
        text_blocks = self._extract_pdf_text_with_pages(pdf_path)
        filename = os.path.basename(pdf_path)
        
        # Get high-quality markdown from docling
        docling_md = self._convert_with_docling(pdf_path)
        
        # Get accurate header information from pymupdf4llm
        header_info = self._extract_header_info_from_pymupdf4llm(pdf_path)
        
        # Merge the two results
        merged_md = self._merge_markdown_with_headers(docling_md, header_info)
        
        # Fallback to pymupdf4llm if docling fails
        if not merged_md:
            merged_md = pymupdf4llm.to_markdown(pdf_path)
        
        # If both fail, use the basic text extracted from blocks
        if not merged_md:
            merged_md = "\n\n".join(block.text for block in text_blocks)
        
        # Organize blocks by page for easier reference
        blocks_by_page = defaultdict(list)
        for block in text_blocks:
            blocks_by_page[block.page_num].append(block)
        
        # Split by headers
        try:
            md_header_splits = self.markdown_splitter.split_text(merged_md)
        except Exception as e:
            print(f"Error splitting markdown: {e}")
            md_header_splits = [{"content": merged_md, "metadata": {}}]
            
        # Process each chunk
        chunks = []
        for header_chunk in md_header_splits:
            # Extract content and metadata based on the type returned
            if isinstance(header_chunk, dict):
                content = header_chunk.get("content", "")
                metadata = header_chunk.get("metadata", {})
            else:  # Assuming it's a Document-like object
                content = header_chunk.page_content if hasattr(header_chunk, 'page_content') else str(header_chunk)
                metadata = header_chunk.metadata if hasattr(header_chunk, 'metadata') else {}
            
            # Find page ranges for this chunk
            page_numbers = self.text_matcher.find_page_ranges(content, text_blocks)
            
            # Prepare chunk metadata
            chunk_metadata = {
                **metadata,
                'filename': filename,
                'page': page_numbers,
                'total_pages': max(block.metadata['total_pages'] for block in text_blocks) if text_blocks else 0,
            }
            
            # Add PDF metadata
            if text_blocks:
                first_block = next(iter(text_blocks))
                chunk_metadata.update({
                    'title': first_block.metadata.get('title', ''),
                    'author': first_block.metadata.get('author', ''),
                })
            
            # Add additional metadata for blocks on these pages
            relevant_blocks = [
                block for block in text_blocks 
                if block.page_num in page_numbers
            ]
            if relevant_blocks:
                chunk_metadata.update({
                    'contains_header': any(block.metadata.get('is_header', False) for block in relevant_blocks),
                    'contains_footer': any(block.metadata.get('is_footer', False) for block in relevant_blocks),
                    'has_images': any(block.metadata.get('has_images', False) for block in relevant_blocks),
                    'has_tables': any(block.metadata.get('has_tables', False) for block in relevant_blocks),
                })
            
            # Add any extra info provided
            if extra_info:
                chunk_metadata.update(extra_info)
                
            # Create document
            doc = Document(
                text=content,
                metadata=chunk_metadata
            )
            chunks.append(doc)
        
        return chunks

/Users/saurabshrestha/Documents/privategpt/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_reader = EnhancedPDFReader(similarity_threshold=0.85)
pdf_path = r"/Users/saurabshrestha/Downloads/AMR Reports/untitled folder/AMR Report 2023.pdf"
documents = pdf_reader.load_data(pdf_path)

documents

FileNotFoundError: no such file: '/Users/saurabshrestha/Downloads/AMR Reports/untitled folder/AMR Report 2023.pdf'